# Model Bayesian Poisson untuk Gol Piala Dunia

Notebook ini mengikuti rencana pada draft teman: data utama adalah Piala Dunia 2014, 2018, dan 2022. Targetnya adalah membangun model Bayesian Poisson sederhana, mengecek kecocokan Poisson secara visual/numerik, lalu menghasilkan distribusi prediktif posterior untuk pertandingan Piala Dunia 2026.

Fokusnya tetap Mathematical and Statistical Foundations: likelihood Poisson, prior Gamma, posterior Gamma, interval kredibel, dan distribusi prediktif posterior. Notebook ini ditulis dalam Bahasa Indonesia agar langsung bisa dipakai untuk bahan presentasi.

## Model Statistik

Misalkan `Y_i` adalah total gol pada pertandingan ke-`i`.

$$Y_i \mid \lambda \sim Poisson(\lambda)$$

Parameter `lambda` menyatakan rata-rata jumlah gol per pertandingan.

Sesuai draft, prior yang dipakai adalah prior Gamma lemah:

$$\lambda \sim Gamma(1, 1)$$

Dengan parameterisasi rate, jika data memiliki total gol $\sum y_i$ dan jumlah pertandingan $n$, maka:

$$\lambda \mid data \sim Gamma(1 + \sum y_i, 1 + n)$$

In [ ]:
from pathlib import Path
import json
import math

ROOT = Path.cwd()
while not (ROOT / "worldcup.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_DIR = ROOT / "worldcup.json"
if not DATA_DIR.exists():
    raise FileNotFoundError("Folder worldcup.json tidak ditemukan dari lokasi kerja notebook.")


def load_worldcup_matches(year):
    path = DATA_DIR / str(year) / "worldcup.json"
    with path.open(encoding="utf-8") as file:
        data = json.load(file)
    return data["matches"]


def scored_matches(year):
    return [
        match for match in load_worldcup_matches(year)
        if "score" in match and ("ft" in match["score"] or "et" in match["score"])
    ]


def score_for_model(match):
    """Gunakan skor akhir termasuk extra time jika tersedia.

    Ini mengikuti angka agregat pada draft: 2014=171, 2018=169, 2022=172.
    Adu penalti tidak dihitung karena bukan gol dalam permainan.
    """
    score = match["score"]
    if "et" in score:
        return score["et"]
    return score["ft"]


def total_goals_by_match(year):
    return [sum(score_for_model(match)) for match in scored_matches(year)]


def frequency_table(values):
    counts = {}
    for value in values:
        counts[value] = counts.get(value, 0) + 1
    return dict(sorted(counts.items()))


def ascii_bar_chart(rows, label_key, value_key, width=32):
    max_value = max(row[value_key] for row in rows) if rows else 0
    for row in rows:
        value = row[value_key]
        bar_length = round((value / max_value) * width) if max_value else 0
        bar = "#" * bar_length
        print(f"{row[label_key]!s:>10} | {bar:<{width}} {value:.3f}")


def gamma_pdf(x, alpha, beta):
    log_density = alpha * math.log(beta) - math.lgamma(alpha) + (alpha - 1) * math.log(x) - beta * x
    return math.exp(log_density)


def poisson_pmf(k, lam):
    return math.exp(-lam + k * math.log(lam) - math.lgamma(k + 1))


def negative_binomial_predictive_pmf(k, alpha, beta):
    """Prediktif posterior P(Y_new = k) untuk model Gamma-Poisson."""
    log_coef = math.lgamma(k + alpha) - math.lgamma(alpha) - math.lgamma(k + 1)
    log_prob = log_coef + alpha * math.log(beta / (beta + 1)) + k * math.log(1 / (beta + 1))
    return math.exp(log_prob)


def credible_interval_from_pmf(rows, mass=0.90):
    lower_tail = (1 - mass) / 2
    upper_tail = 1 - lower_tail
    cumulative = 0.0
    lower = None
    upper = None
    for row in rows:
        cumulative += row["probability"]
        if lower is None and cumulative >= lower_tail:
            lower = row["goals"]
        if upper is None and cumulative >= upper_tail:
            upper = row["goals"]
            break
    return lower, upper

## Data: Piala Dunia 2014, 2018, dan 2022

Draft memakai tiga edisi terakhir yang sudah selesai: 2014, 2018, dan 2022. Notebook ini menghitung total gol dari skor akhir pertandingan. Jika pertandingan masuk extra time, skor `et` dipakai; jika tidak, skor `ft` dipakai. Adu penalti tidak dihitung.

Dengan aturan ini, total gol cocok dengan draft: 171, 169, dan 172 gol.

In [ ]:
analysis_years = [2014, 2018, 2022]

summary_rows = []
train_goals = []
for year in analysis_years:
    goals = total_goals_by_match(year)
    train_goals.extend(goals)
    summary_rows.append({
        "year": year,
        "matches": len(goals),
        "total_goals": sum(goals),
        "avg_goals": sum(goals) / len(goals),
    })

for row in summary_rows:
    print(
        f"{row['year']}: {row['matches']} pertandingan, "
        f"{row['total_goals']} gol, rata-rata = {row['avg_goals']:.3f}"
    )

print("\nRata-rata gol per edisi:")
ascii_bar_chart(summary_rows, "year", "avg_goals")

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 4))
    plt.bar([str(row["year"]) for row in summary_rows], [row["avg_goals"] for row in summary_rows])
    plt.title("Rata-rata Gol per Pertandingan")
    plt.xlabel("Piala Dunia")
    plt.ylabel("Rata-rata gol")
    plt.show(block=False)
    plt.close()
except ImportError:
    print("matplotlib tidak tersedia; grafik teks ditampilkan sebagai pengganti.")

## Update Bayesian Gamma-Poisson

Sesuai draft, prior utama adalah:

$$\lambda \sim Gamma(1,1)$$

Data memiliki 192 pertandingan dan 512 total gol, sehingga posterior menjadi:

$$\lambda \mid data \sim Gamma(513,193)$$

Posterior mean mendekati rata-rata empiris, tetapi tetap dinyatakan sebagai distribusi sehingga ketidakpastian parameter masih terlihat.

In [ ]:
alpha_prior = 1.0
beta_prior = 1.0

train_total_goals = sum(train_goals)
train_match_count = len(train_goals)

alpha_post = alpha_prior + train_total_goals
beta_post = beta_prior + train_match_count

empirical_mean = train_total_goals / train_match_count
prior_mean = alpha_prior / beta_prior
posterior_mean = alpha_post / beta_post
posterior_variance = alpha_post / (beta_post ** 2)
posterior_sd = math.sqrt(posterior_variance)
normal_ci_95 = (posterior_mean - 1.96 * posterior_sd, posterior_mean + 1.96 * posterior_sd)

print(f"Total pertandingan: {train_match_count}")
print(f"Total gol: {train_total_goals}")
print(f"Rata-rata empiris: {empirical_mean:.3f}")
print(f"Prior: Gamma({alpha_prior:.0f}, {beta_prior:.0f})")
print(f"Posterior: Gamma({alpha_post:.0f}, {beta_post:.0f})")
print(f"Posterior mean: {posterior_mean:.3f}")
print(f"Posterior variance: {posterior_variance:.4f}")
print(f"Posterior standard deviation: {posterior_sd:.3f}")
print(f"Aproksimasi 95% credible interval: [{normal_ci_95[0]:.2f}, {normal_ci_95[1]:.2f}] gol")

try:
    import matplotlib.pyplot as plt
    import numpy as np
    xs = np.linspace(0.01, 5, 400)
    prior_density = [gamma_pdf(float(x), alpha_prior, beta_prior) for x in xs]
    posterior_density = [gamma_pdf(float(x), alpha_post, beta_post) for x in xs]
    plt.figure(figsize=(7, 4))
    plt.plot(xs, prior_density, label="Prior Gamma(1,1)", linestyle="--")
    plt.plot(xs, posterior_density, label="Posterior Gamma(513,193)")
    plt.title("Prior vs Posterior untuk λ")
    plt.xlabel("λ: rata-rata gol per pertandingan")
    plt.ylabel("Densitas")
    plt.legend()
    plt.show(block=False)
    plt.close()
except ImportError:
    print("matplotlib/numpy tidak tersedia; ringkasan posterior ditampilkan di atas.")

## Distribusi Frekuensi Gol dan Goodness-of-Fit

Draft merekomendasikan distribusi frekuensi gol dan goodness-of-fit. Di sini kita membandingkan frekuensi aktual dengan frekuensi yang diharapkan oleh distribusi Poisson dengan parameter posterior mean.

Untuk menjaga notebook tetap ringan, ukuran kecocokan yang ditampilkan adalah statistik chi-square deskriptif. Interpretasinya: semakin kecil selisih aktual dan expected, semakin masuk akal model Poisson sebagai pendekatan agregat.

In [ ]:
observed_frequency = frequency_table(train_goals)
max_goal = max(max(observed_frequency), 9)
observed_grouped = []
for k in range(0, 9):
    observed_grouped.append(observed_frequency.get(k, 0))
observed_grouped.append(sum(count for goal, count in observed_frequency.items() if goal >= 9))

expected_grouped = []
for k in range(0, 9):
    expected_grouped.append(train_match_count * poisson_pmf(k, posterior_mean))
expected_grouped.append(train_match_count * (1 - sum(poisson_pmf(k, posterior_mean) for k in range(0, 9))))

labels = [str(k) for k in range(0, 9)] + ["9+"]
goodness_of_fit_rows = []
chi_square_stat = 0.0
for label, observed, expected in zip(labels, observed_grouped, expected_grouped):
    contribution = ((observed - expected) ** 2 / expected) if expected > 0 else 0.0
    chi_square_stat += contribution
    goodness_of_fit_rows.append({
        "goals": label,
        "observed": observed,
        "expected": expected,
        "chi_square": contribution,
    })

print("gol | aktual | expected Poisson | kontribusi chi-square")
for row in goodness_of_fit_rows:
    print(f"{row['goals']:>3} | {row['observed']:>6} | {row['expected']:>16.2f} | {row['chi_square']:.3f}")
print(f"\nStatistik chi-square deskriptif: {chi_square_stat:.3f}")

try:
    import matplotlib.pyplot as plt
    x = list(range(len(labels)))
    plt.figure(figsize=(8, 4))
    plt.bar([v - 0.2 for v in x], observed_grouped, width=0.4, label="Aktual")
    plt.bar([v + 0.2 for v in x], expected_grouped, width=0.4, label="Expected Poisson")
    plt.xticks(x, labels)
    plt.title("Frekuensi Gol Aktual vs Poisson")
    plt.xlabel("Total gol dalam pertandingan")
    plt.ylabel("Jumlah pertandingan")
    plt.legend()
    plt.show(block=False)
    plt.close()
except ImportError:
    print("matplotlib tidak tersedia; tabel goodness-of-fit ditampilkan di atas.")

## Distribusi Prediktif Posterior untuk 2026

Dengan posterior Gamma(513,193), prediksi untuk satu pertandingan baru tidak hanya memakai satu nilai λ. Ketidakpastian λ ikut diintegrasikan melalui distribusi prediktif posterior.

Untuk model Gamma-Poisson, distribusi prediktif ini berbentuk negative binomial. Bagian ini menjawab pertanyaan draft: berapa peluang pertandingan Piala Dunia berikutnya menghasilkan 0, 1, 2, 3, dan seterusnya gol?

In [ ]:
predictive_rows = []
for k in range(0, 10):
    predictive_rows.append({
        "goals": k,
        "probability": negative_binomial_predictive_pmf(k, alpha_post, beta_post),
    })
prob_10_plus = 1 - sum(row["probability"] for row in predictive_rows)
interval_90 = credible_interval_from_pmf(predictive_rows + [{"goals": 10, "probability": prob_10_plus}], mass=0.90)

print("gol | probabilitas prediksi")
for row in predictive_rows:
    print(f"{row['goals']:>3} | {row['probability']:.4f}")
print(f"10+ | {prob_10_plus:.4f}")
print(f"\nInterval prediktif posterior 90% untuk satu pertandingan 2026: {interval_90[0]} sampai {interval_90[1]} gol")

prob_0_to_1 = sum(row["probability"] for row in predictive_rows if row["goals"] <= 1)
prob_2_to_3 = sum(row["probability"] for row in predictive_rows if row["goals"] in [2, 3])
prob_4_plus = 1 - sum(row["probability"] for row in predictive_rows if row["goals"] <= 3)
print(f"P(0-1 gol): {prob_0_to_1:.3f}")
print(f"P(2-3 gol): {prob_2_to_3:.3f}")
print(f"P(4+ gol): {prob_4_plus:.3f}")

try:
    import matplotlib.pyplot as plt
    labels_plot = [str(row["goals"]) for row in predictive_rows] + ["10+"]
    probs_plot = [row["probability"] for row in predictive_rows] + [prob_10_plus]
    plt.figure(figsize=(8, 4))
    plt.bar(labels_plot, probs_plot)
    plt.title("Distribusi Prediktif Posterior 2026")
    plt.xlabel("Total gol dalam satu pertandingan")
    plt.ylabel("Probabilitas")
    plt.show(block=False)
    plt.close()
except ImportError:
    print("matplotlib tidak tersedia; probabilitas prediksi ditampilkan di atas.")

## Catatan Pengembangan

Model ini sengaja dibuat agregat agar konsep Bayesian Poisson jelas. Batasannya sama seperti yang disebutkan dalam draft:

- semua pertandingan memakai satu rata-rata gol yang sama;
- kekuatan tim belum dimodelkan;
- fase grup dan fase gugur belum dibedakan;
- skor rendah seperti 0-0 dan 1-1 dapat membutuhkan koreksi khusus seperti Dixon-Coles;
- model hierarkis berbasis kekuatan tim dapat menjadi pengembangan lanjutan.

Dengan demikian, notebook ini mengikuti draft sebagai model dasar, lalu menambahkan implementasi data lengkap, goodness-of-fit, visualisasi, dan distribusi prediktif posterior untuk 2026.